# Financial News Sentiment Benchmarking Hub

This notebook implements and benchmarks three model architectures on the 3-class financial sentiment classification task: 
1. **Simple RNN** (Word Embeddings -> RNN -> Mean Pooling -> Dense Classifier)
2. **LSTM** (Word Embeddings -> LSTM -> Mean Pooling -> Dense Classifier)
3. **FinBERT** (Fine-tuning the pre-trained `ProsusAI/finbert` transformer)

---

## 1. Google Colab & Environment Setup

If you are running in Google Colab, mount Google Drive to save checkpoints and install dependencies.

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone repository or navigate to workspace directory on Drive
    # Replace with your actual directory path on Google Drive if needed
    # %cd /content/drive/MyDrive/financial-news-sentiment
    
    # Install dependencies
    print("Installing dependencies...")
    !pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn pyyaml streamlit
else:
    print("Running in Local environment.")

### Path Configuration

Ensure `src` package is in Python search path.

In [ ]:
# Make the repository root importable from the notebook.\nfrom pathlib import Path\nPROJECT_ROOT = Path.cwd()\nif PROJECT_ROOT.name == 'notebooks':\n    PROJECT_ROOT = PROJECT_ROOT.parent\nimport sys\nsys.path.insert(0, str(PROJECT_ROOT))\nprint('Project root:', PROJECT_ROOT)

## 2. Phase 1 — Dataset & EDA

We load the `zeroshot/twitter-financial-news-sentiment` dataset, split sizes, analyze class distribution, compute tweet lengths, and check for leakage overlap.

In [ ]:
from src.phase1_eda import run_eda
run_eda()

## 3. Phase 2 — Preprocessing

The preprocessing module cleans URLs, user mentions, preserves tickers (e.g., `$AAPL`), and constructs the vocabulary exclusively from training data.

In [ ]:
from src.preprocessing import clean_text, tokenize, Vocabulary

sample_text = "Check out $AAPL and $TSLA! High gains today at http://example.com @user."
print("Original: ", sample_text)
print("Cleaned:  ", clean_text(sample_text))
print("Tokens:   ", tokenize(clean_text(sample_text)))

## 4. Phase 3 — Simple RNN Baseline

Train a 2-layer simple RNN classifier on CPU/GPU.

In [ ]:
!python ../src/training/train_rnn_lstm.py --model rnn

## 5. Phase 4 — LSTM Baseline

Train a 2-layer LSTM classifier on CPU/GPU.

In [ ]:
!python ../src/training/train_rnn_lstm.py --model lstm

## 6. Phase 5 — FinBERT Fine-Tuning (Colab GPU Recommended)

Fine-tune pre-trained `ProsusAI/finbert` using Hugging Face's `Trainer` API.

In [ ]:
# If running locally on CPU, this might take a very long time.
# Training time depends on hardware and runtime conditions.
!python ../src/training/train_finbert.py

## 7. Phase 7 — Model Comparison

Compare validation metrics and plot accuracies/F1s.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

log_path = "../reports/experiment_log.csv"
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    print("--- Experiment Benchmarks ---")
    display(df)
    
    # Visualize Accuracy and F1
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.barplot(data=df, x="model", y="val_accuracy", palette="viridis")
    plt.title("Validation Accuracy Comparison")
    plt.ylim(0, 1.0)
    
    plt.subplot(1, 2, 2)
    sns.barplot(data=df, x="model", y="val_macro_f1", palette="viridis")
    plt.title("Validation Macro F1 Comparison")
    plt.ylim(0, 1.0)
    
    plt.tight_layout()
    plt.savefig("../figures/model_comparison.png")
    plt.show()
else:
    print("No log file found. Train the models first!")

## 8. View Confusion Matrices

Compare visual class confusions.

In [ ]:
from IPython.display import Image, display

fig_dir = "../figures"
for m in ["RNN", "LSTM", "FINBERT"]:
    path = os.path.join(fig_dir, f"confusion_matrix_{m.lower()}.png")
    if os.path.exists(path):
        print(f"=== {m} Confusion Matrix ===")
        display(Image(filename=path))
        print("\n")

## 9. Final Validation-Set Class Distribution\n\nThe supplied validation split contains **2,388 samples**. The class distribution is reported below without resampling or synthetic expansion.\n

In [ ]:
from src.data_loader import load_data\nimport pandas as pd\n\n_, val_df = load_data()\nlabel_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}\nval_counts = pd.to_numeric(val_df['label'], errors='coerce').map(label_map).value_counts().reindex(\n    ['Bearish', 'Bullish', 'Neutral'], fill_value=0\n)\ndisplay(val_counts.rename('count').to_frame())\nprint('Validation samples:', int(val_counts.sum()))

## 10. Final Quantitative Results\n\nThe final benchmark on the supplied validation split is:\n\n| Model | Validation Accuracy | Macro F1 | Macro Precision | Macro Recall |\n|---|---:|---:|---:|---:|\n| Simple RNN | 78.48% | 66.54% | 69.50% | 64.67% |\n| LSTM | 81.24% | 72.14% | 74.66% | 70.36% |\n| FinBERT | 87.77% | 83.72% | See report | See report |\n\nThe LSTM improved over the simple RNN by **2.76 percentage points in accuracy** and **5.60 percentage points in macro F1**. FinBERT further improved validation accuracy and macro F1 relative to both recurrent baselines.\n

## 11. Class-wise Evaluation and Error Analysis\n\nThe project saves class-wise precision, recall, F1, and support in `reports/classification_report_*.csv`, and representative validation errors in `reports/error_analysis_*.csv`. Confusion matrices are saved under `figures/`. These artifacts provide the required class-wise evaluation and error-analysis evidence without modifying the supplied train/validation split.

In [ ]:
from pathlib import Path\n\nfor model in ['rnn', 'lstm', 'finbert']:\n    report_path = PROJECT_ROOT / 'reports' / f'classification_report_{model}.csv'\n    error_path = PROJECT_ROOT / 'reports' / f'error_analysis_{model}.csv'\n    if report_path.exists():\n        print(f'=== {model.upper()} class-wise report ===')\n        display(pd.read_csv(report_path))\n    if error_path.exists():\n        errors = pd.read_csv(error_path)\n        print(f'{model.upper()} validation errors saved:', len(errors))\n        display(errors.head(5))

## 12. Final Interpretation\n\n- The simple RNN establishes the mandatory recurrent baseline.\n- The LSTM provides a stronger recurrent baseline on the validation set, with higher accuracy and macro F1 than the simple RNN.\n- FinBERT provides the strongest measured validation metrics in this experiment, demonstrating the benefit of a finance-oriented pretrained transformer for this task.\n- The validation distribution is imbalanced toward Neutral, so macro F1 and class-wise metrics are reported alongside accuracy.\n- No train/validation resampling, duplication, or synthetic padding is used. The project evaluates the actual supplied splits.\n\nThe Streamlit application uses the trained checkpoints for live three-class inference and presents prediction probabilities, benchmark metrics, validation-set class distribution, and confusion matrices.